Smoothing: Shortcuts for Planned Paths
======================================

**Topic**: Smoothing of collision-free paths with shortcuts - the generic approach and its three strategies

Version | Author | Remark
------------ | ------------- | -------------
0.1 | Björn Hein | Generic approach, three strategies, pitfalls
0.2 | Björn Hein | Robots: reworked on the basis of IP-10-0 (work space and joint space, safety margin, more joints)


This worksheet **explains the generic approach for path smoothing** of the lecture: a path from a previous planning is shortened step by step by shortcuts, every new segment is checked with the collision checker of the planning. Three strategies choose the shortcuts: pure probabilistic, extended probabilistic (Latombe) and deterministic (Berchtold & Glavina).

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-4-0-Test-Environments (collision checker, benchmarks, `IPPerfMonitor`) and at least one of the planners, e.g. IP-5-2-PRM-KClosest, IP-7-0-PRM-Lazy or IP-9-0-RRT-Basics
* For the last chapter: IP-10-0-PlanarManipulator (planar robot, joint space, resolution of the line check)

**Dependencies**:

* IPEnvironment.py (`CollisionChecker.lineInCollision`), IPSmoothing.py (the class `Smoother`), IPVISSmoothing.py (figures of this notebook), IPTestSuite.py (benchmarks) and the planners IPKClosestPRM.py, IPRRT.py, IPLazyPRM.py, IPAStar.py
* For the robots: IPPlanarManipulator.py, IPEnvironmentKin.py (`KinChainCollisionChecker`) and IPVISPlanarManipulator.py (figures of the robot in the work space)

Important links are:

* General Info: https://www.python.org
* NetworkX: https://networkx.org
* NumPy: https://numpy.org/doc/stable/
* Matplotlib: https://matplotlib.org/stable/
* Shapely: https://shapely.readthedocs.io/

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Idea

The planners so far (PRM, RRT, A\*, ...) give **collision-free** paths, but they do not look nice: detours, zigzag, many unnecessary points. The reason is the random sampling (PRM, RRT) or the grid (A\*).

Smoothing improves such a path afterwards: a part of the path is replaced by a **shortcut**, a straight segment. The shortcut has to be collision-free, so every new segment is checked with the collision checker of the planning - **one check of a segment is one step** of the smoothing and the measure of its effort.

The **generic approach** of the lecture consists of three parts:

* the path $p(l)$ with the position $l \in [0, 1]$ along the path: $p(0) = q_{start}$, $p(1) = q_{goal}$
* `OptimizationStrategy(l_A, l_B, p)`: if the segment from $p(l_A)$ to $p(l_B)$ is collision-free, the path between them is replaced by this segment
* `ChooseShortcut(p, l_A, l_B, success)`: the **selection function** chooses the next positions $l_A$, $l_B$ - here the three strategies differ

```
Smooth(p):
    (l_A, l_B) <- ChooseShortcut(p, 0, 1, true)
    while l_A != l_B:
        success <- OptimizationStrategy(l_A, l_B, p)
        (l_A, l_B) <- ChooseShortcut(p, l_A, l_B, success)
    return p
```

# A path from a previous planning

Smoothing starts with the result of a planner. All planners of the course return the solution as list of nodes of `planner.graph`, every node has the attribute `pos`. `pathFromPlanner` of IPSmoothing.py turns it into a list of configurations. The collision checker of the planning is `planner._collisionChecker` (or `benchmark.collisionChecker`).

Example: the K-Closest-PRM on the benchmark "MyField". `IPPerfMonitor` counts the calls of `lineInCollision` during the planning.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
%matplotlib inline

import IPTestSuite as ts
import IPVISSmoothing as vis
from IPKClosestPRM import KClosestPRM
from IPPerfMonitor import IPPerfMonitor
from IPSmoothing import pathFromPlanner, pathLength
from IPVISBasicPRM import basicPRMVisualize


def lineChecks():
    # number of checked segments (lineInCollision, lineInCollisionExact) since the last IPPerfMonitor.clearData()
    calls = IPPerfMonitor.dataFrame()
    return int(calls["name"].isin(["lineInCollision", "lineInCollisionExact"]).sum()) if len(calls) else 0


benchmark = ts.benchList[3]   # "MyField"
kClosestConfig = {"k": 5, "numNodes": 300}

random.seed(1)
IPPerfMonitor.clearData()
planner = KClosestPRM(benchmark.collisionChecker)
solution = planner.planPath(benchmark.startList, benchmark.goalList, kClosestConfig)
planningChecks = lineChecks()

path = pathFromPlanner(planner, solution)
print(f"K-Closest-PRM on {benchmark.name}: {planner.graph.number_of_nodes()} nodes, {planningChecks} line checks")
print(f"path: {len(path)} points, length {pathLength(path):.1f}, straight line {math.dist(path[0], path[-1]):.1f}")

vis.showPlannerResult(planner, solution, basicPRMVisualize, path, f"K-Closest-PRM on {benchmark.name}")

# The path $p(l)$ and a shortcut

## Positions on the path

A position $l \in [0, 1]$ is the length along the path from the start, normalized by the length of the path. The path points are $P_i = p(l_i)$, but $l_A$ and $l_B$ may lie anywhere - also inside a segment.

The example of the lecture: the lengths of the segments are chosen such that the positions are exactly the ones of the slides.

In [ ]:
from IPEnvironment import CollisionChecker

EPS = 1e-9


def positions(path):
    # position l_i in [0, 1] of every path point: length from the start, normalized by the path length
    total = pathLength(path)
    ls, length = [0.0], 0.0
    for a, b in zip(path[:-1], path[1:]):
        length += math.dist(a, b)
        ls.append(length / total if total > 0 else 0.0)
    return ls


def pointAt(path, l, ls=None):
    # configuration p(l) and the index k of the segment (path[k], path[k+1]) it lies on
    ls = ls or positions(path)
    for k in range(len(path) - 1):
        if l <= ls[k + 1] or k == len(path) - 2:
            span = ls[k + 1] - ls[k]
            t = 0.0 if span <= 0 else min(max((l - ls[k]) / span, 0.0), 1.0)
            return tuple(a + t * (b - a) for a, b in zip(path[k], path[k + 1])), k


scene, limits, lecturePath = vis.lectureExample()
lectureChecker = CollisionChecker(scene, limits=limits)

print("positions l_i:", [round(l, 2) for l in positions(lecturePath)])
print("p(0.3) =", tuple(round(c, 2) for c in pointAt(lecturePath, 0.3)[0]), "on segment", pointAt(lecturePath, 0.3)[1])
vis.showPath(lectureChecker, lecturePath, "path of the lecture, $P_i = p(l_i)$", labels=True)

## OptimizationStrategy

`OptimizationStrategy(l_A, l_B, p)` checks the segment from $p(l_A)$ to $p(l_B)$. If it is collision-free, all path points with $l_A < l_i < l_B$ are removed and $p(l_A)$, $p(l_B)$ are inserted.

If there is no path point between $l_A$ and $l_B$, the segment is a part of the path anyway - there is nothing to shorten. Then no check is made and the function returns *false*.

In [ ]:
def optimizationStrategy(collisionChecker, path, lA, lB):
    # shortcut from p(l_A) to p(l_B); returns (success, new path, checked)
    ls = positions(path)
    inner = [i for i in range(1, len(path) - 1) if lA < ls[i] < lB]   # path points between l_A and l_B
    if not inner:
        return False, path, False                                     # nothing to shorten: no check
    qa, _ = pointAt(path, lA, ls)
    qb, _ = pointAt(path, lB, ls)
    if collisionChecker.lineInCollision(list(qa), list(qb)):
        return False, path, True
    head, tail = path[:inner[0]], path[inner[-1] + 1:]
    newPath = (head + ([qa] if math.dist(qa, head[-1]) > EPS else [])
               + ([qb] if math.dist(qb, tail[0]) > EPS else []) + tail)
    return True, newPath, True


success, newPath, checked = optimizationStrategy(lectureChecker, lecturePath, 0.3, 0.55)
print(f"shortcut (0.3, 0.55): success {success}, P3 and P4 removed, p(0.3) and p(0.55) inserted, "
      f"length {pathLength(lecturePath):.1f} -> {pathLength(newPath):.1f}")
vis.showShortcut(lectureChecker, lecturePath, 0.3, 0.55, newPath, success)

## The basic smoothing algorithm

`smooth` is `Smooth(p)` of the slides. The selection function is a parameter, it gets a dictionary `state` as memory between its calls (e.g. for $l_{rand}$). Every tested shortcut is recorded for the figures. As the loop condition $l_A \neq l_B$ is hardly ever reached by the probabilistic strategies, the number of collision checks is limited by `maxChecks`.

In [ ]:
def smooth(collisionChecker, path, chooseShortcut, maxChecks=100):
    # Smooth(p): returns (smoothed path, list of the tested shortcuts)
    state = dict()                     # memory of the selection function
    steps = []
    lA, lB = chooseShortcut(path, 0.0, 1.0, True, state)
    iterations = 0
    while lA != lB and len(steps) < maxChecks and iterations < 100 * maxChecks:
        iterations += 1
        previous = path
        success, path, checked = optimizationStrategy(collisionChecker, path, lA, lB)
        if checked:
            steps.append(dict(lA=lA, lB=lB, qa=pointAt(previous, lA)[0], qb=pointAt(previous, lB)[0], free=success,
                              previous=previous, path=path, strategy=state.get("strategy"),
                              chosen=state.get("chosen"), lRand=state.get("lRand"), worst=state.get("worst")))
        lA, lB = chooseShortcut(path, lA, lB, success, state)
    return path, steps

# Strategy 1: pure probabilistic

Two positions are chosen at random. The arguments of the selection function are not used - they are only given for the same interface as the other strategies. A shortcut, which collides, is simply discarded.

The first five checks on the example of the lecture (the same seed as for the slides, so the steps are the ones of the slides):

In [ ]:
def chooseRandom(path, lA, lB, success, state):
    # pure probabilistic: two random positions (the arguments are not used)
    state["strategy"], state["chosen"] = "random", "new"
    x, y = random.random(), random.random()
    return min(x, y), max(x, y)


random.seed(38)
randomPath, randomSteps = smooth(lectureChecker, lecturePath, chooseRandom, maxChecks=5)
vis.showSteps(lectureChecker, randomSteps, lecturePath)

Each shortcut starts and ends somewhere inside a segment: there **new points** are inserted. The pure probabilistic strategy does not care, how promising a shortcut is.

# Strategy 2: extended probabilistic (Latombe)

A random position $l_{rand}$ is chosen and a shortcut around it, from the middle between start and $l_{rand}$ to the middle between $l_{rand}$ and goal. If the shortcut collides, it is not discarded: the interval is **halved towards $l_{rand}$** and tested again. A new $l_{rand}$ is chosen after a success or as soon as the interval is shorter than $\Delta l_{min}$. $l_{rand}$ is kept between the calls (in `state`).

```
ChooseShortcut(p, l_A, l_B, success):
    if success or l_B - l_A < Δl_min:
        l_rand <- random(0, 1)
        l_A <- 0;  l_B <- 1
    l_A <- (l_A + l_rand) / 2
    l_B <- (l_B + l_rand) / 2
    return (l_A, l_B)
```

**Remember**: on the original slides the condition read $l_A - l_B < \Delta l_{min}$. As $l_A < l_B$ this is always true - the interval would never be halved.

In [ ]:
def chooseLatombe(path, lA, lB, success, state, deltaLMin=0.02):
    # extended probabilistic: l_rand is kept in state, the interval around it is halved
    state["strategy"], state["chosen"] = "latombe", "halve"
    if success or "lRand" not in state or lB - lA < deltaLMin:
        state["lRand"] = random.random()
        lA, lB = 0.0, 1.0
        state["chosen"] = "new"
    return (lA + state["lRand"]) / 2, (lB + state["lRand"]) / 2


random.seed(51)
latombePath, latombeSteps = smooth(lectureChecker, lecturePath, chooseLatombe, maxChecks=6)
vis.showSteps(lectureChecker, latombeSteps, lecturePath)

# Strategy 3: deterministic (Berchtold & Glavina)

## Rating of the path points

Every inner path point is rated by the detour it causes:

$Q_{p_i} = \dfrac{|p_{i-1} p_i| + |p_i p_{i+1}|}{|p_{i-1} p_{i+1}|} \ge 1$

$Q = 1$: the point lies on the straight line between its neighbours, there is nothing to gain. The larger $Q$, the sharper the spike or the longer the detour - the **worst point** has the highest $Q$ and is treated first. In practice the points are kept in a priority list (constant time for the worst one); here a simple search over all points is sufficient.

In [ ]:
def quality(path, i):
    # rating Q of the inner path point i
    base = math.dist(path[i - 1], path[i + 1])
    detour = math.dist(path[i - 1], path[i]) + math.dist(path[i], path[i + 1])
    return math.inf if base < EPS else detour / base


print("Q:", [round(quality(lecturePath, i), 2) for i in range(1, len(lecturePath) - 1)])
vis.showQualities(lectureChecker, lecturePath, "rating $Q$ of the inner path points")

## Selection function

* After a success (and at the beginning): the worst point $p(l_w)$ is chosen and its predecessor and successor are connected, i.e. the worst point is removed.
* If this connection collides, the interval is **halved towards $p(l_w)$**: $l_A \leftarrow \frac{1}{2}(l_A + l_w)$, $l_B \leftarrow \frac{1}{2}(l_B + l_w)$. These are the new points of the operation **DelTree** of the slides: $p_{z1} = p_2 + \overrightarrow{p_2 p_1} / 2^t$, $p_{z2} = p_2 + \overrightarrow{p_2 p_3} / 2^t$ with $t = 1, 2, \dots$
* If even the interval $\Delta l_{min}$ collides, the point is skipped. It gets a new chance only when one of its neighbours has changed - otherwise the same point would be chosen again and again.
* No candidate left: the function returns $l_A = l_B$ and `Smooth` ends by itself.

In [ ]:
def chooseDeterministic(path, lA, lB, success, state, deltaLMin=0.02):
    # deterministic (Berchtold & Glavina): worst path point first, if necessary bisection towards it (DelTree)
    state["strategy"] = "deterministic"
    failed = state.setdefault("failed", set())      # points, whose bisection failed
    worst = state.get("worst")
    if success and "shortcut" in state:
        # the neighbours of the new segment have a new rating: they get a new chance
        for q in state["shortcut"]:
            i = min(range(len(path)), key=lambda k: math.dist(path[k], q))
            for j in (i - 1, i, i + 1):
                if 0 <= j < len(path):
                    failed.discard(path[j])
    ls = positions(path)
    if success or worst not in path or lB - lA < deltaLMin:
        if not success and worst in path:
            failed.add(worst)                        # bisection down to deltaLMin failed
        candidates = [i for i in range(1, len(path) - 1) if path[i] not in failed and quality(path, i) > 1 + EPS]
        if not candidates:
            state["worst"] = None
            return 0.0, 0.0                          # l_A = l_B: nothing left, Smooth ends
        i = max(candidates, key=lambda k: quality(path, k))
        state["worst"], state["chosen"] = path[i], "new"
        lA, lB = ls[i - 1], ls[i + 1]                # predecessor and successor
    else:
        lw = ls[path.index(worst)]
        state["chosen"] = "halve"
        lA, lB = (lA + lw) / 2, (lB + lw) / 2       # halve towards the worst point
    state["shortcut"] = (pointAt(path, lA, ls)[0], pointAt(path, lB, ls)[0])
    return lA, lB


deterministicPath, deterministicSteps = smooth(lectureChecker, lecturePath, chooseDeterministic, maxChecks=6)
vis.showSteps(lectureChecker, deterministicSteps, lecturePath, qualities=True)

Check 1 and 2: the values of the path points do not get constantly better. After removing $P_3$, its neighbour $P_4$ is worse than before ($Q$: 1.18 → 2.11). If the algorithm stopped now, the path would be shorter, but $P_4$ worse rated than at the beginning - as it continues, $P_4$ is the next one to be removed. Check 3 and 4: the direct connection collides, the bisection succeeds - the worst point is replaced by two points.

# The class Smoother

IPSmoothing.py contains the same algorithm as class `Smoother`, with an interface similar to the planners:

* `smoothPath(path, config)` smoothes a list of configurations,
* `smoothSolution(planner, solution, config)` smoothes the result of a previous planning: the path is taken from `planner.graph`, the collision checker from the planner (if the smoother has none).

Both return the smoothed path as list of nodes of `smoother.graph` (attribute `pos`), so it can be visualized and animated like the result of a planner. The tested shortcuts are recorded in `smoother.steps`, length and number of points after every check in `smoother.history`. Random numbers come from the module `random`, as for the planners.

With the same seeds it makes exactly the same steps as the functions above:

In [ ]:
from IPSmoothing import Smoother
from IPVISSmoothing import smoothingVisualize

for strategy, choose, seed, checks in (("random", chooseRandom, 38, 5), ("latombe", chooseLatombe, 51, 6),
                                       ("deterministic", chooseDeterministic, 0, 6)):
    random.seed(seed)
    notebookPath, notebookSteps = smooth(lectureChecker, lecturePath, choose, maxChecks=checks)
    random.seed(seed)
    smoother = Smoother(lectureChecker)
    smoother.smoothPath(lecturePath, {"strategy": strategy, "maxChecks": checks})
    same = len(notebookPath) == len(smoother.path) and all(
        math.dist(a, b) < 1e-9 for a, b in zip(notebookPath, smoother.path))
    print(f"{strategy:14s} same path as smooth(): {same}, checks:",
          " ".join(("free" if s["free"] else "coll") + ("/halved" if s["chosen"] == "halve" else "")
                   for s in smoother.steps))

## Smoothing the planned path

The three strategies on the path of the K-Closest-PRM from the beginning, each with 100 checks. Without further configuration `Smoother` checks the shortcuts with `lineInCollision` of the collision checker - as the planners do. In the figures the segments, which collide when checked **exactly** (`lineInCollisionExact`), are drawn orange.

In [ ]:
from IPSmoothing import collidingSegments

smoothConfig = {"maxChecks": 100, "deltaLMin": 0.02}
exact = benchmark.collisionChecker.lineInCollisionExact      # exact check of a segment (point robot, Shapely)

results, smoothers = [], {}
for strategy in ("random", "latombe", "deterministic"):
    random.seed(2)
    IPPerfMonitor.clearData()
    smoother = Smoother()
    smoothed = smoother.smoothSolution(planner, solution, {**smoothConfig, "strategy": strategy})
    smoothers[strategy] = smoother
    checks = lineChecks()
    colliding = collidingSegments(smoother.path, exact)
    print(f"{vis.STRATEGY_NAMES[strategy]:36s} length {pathLength(smoother.path):5.1f}, {len(smoother.path):2d} points, "
          f"{checks} line checks, segments in collision (exact check): {colliding}")
    results.append((vis.STRATEGY_NAMES[strategy], smoother, smoothed))

vis.showResults(results, panelWidth=4.6, check=exact)

**Try it** (optional, needs `ipywidgets`): step through the deterministic smoothing check by check.

In [ ]:
from ipywidgets import IntSlider, interact

smoother = smoothers["deterministic"]
interact(lambda check: vis.plotStep(benchmark.collisionChecker, smoother.steps, smoother.inputPath, check),
         check=IntSlider(min=0, max=len(smoother.steps), value=0));

# Pitfall: checking the shortcuts

`lineInCollision` of `CollisionChecker` checks 40 configurations on a segment, independent of its length. For the short edges of a planner this is mostly fine. But the shortcuts of the smoothing are **long** - on a shortcut of length 20 the checked configurations are 0.5 apart - and they run **close along the obstacles**: the smoothing tightens the path like a rubber band. Exactly there a sampled check misses the corners of the obstacles. The smoothing even searches for these gaps: a shortcut, which cuts a corner a little, is shorter.

How often does this happen? Paths of two planners (RRT and K-Closest-PRM, 5 seeds each) on all benchmarks of IPTestSuite.py, every path smoothed twice with every strategy - once with `lineInCollision`, once with the exact check. Then every smoothed path is checked exactly. (This cell takes some seconds.)

In [ ]:
import statistics

import pandas as pd
from IPRRT import RRT

rrtConfig = {"numberOfGeneratedNodes": 1500, "testGoalAfterNumberOfNodes": 10}
planners = {"RRT": (RRT, rrtConfig), "K-Closest-PRM": (KClosestPRM, kClosestConfig)}
strategies = ("random", "latombe", "deterministic")

tasks = []                 # paths of previous plannings
for bench in ts.benchList:
    for plannerName, (Planner, config) in planners.items():
        for seed in range(5):
            random.seed(seed)
            taskPlanner = Planner(bench.collisionChecker)
            taskSolution = taskPlanner.planPath(bench.startList, bench.goalList, config)
            if taskSolution:
                tasks.append(dict(benchmark=bench, planner=plannerName, path=pathFromPlanner(taskPlanner, taskSolution)))

inputColliding = sum(bool(collidingSegments(task["path"], task["benchmark"].collisionChecker.lineInCollisionExact))
                     for task in tasks)
print(f"{len(tasks)} planned paths, {inputColliding} of them already in collision (exact check)")

rows = []
for strategy in strategies:
    row = {"strategy": vis.STRATEGY_NAMES[strategy]}
    for checkName in ("lineInCollision", "exact"):
        colliding = 0
        for task in tasks:
            taskExact = task["benchmark"].collisionChecker.lineInCollisionExact
            for run in range(2):
                random.seed(run)
                smoother = Smoother(task["benchmark"].collisionChecker)
                config = {"strategy": strategy, "maxChecks": 100}
                if checkName == "exact":
                    config["lineCheck"] = taskExact
                smoother.smoothPath(task["path"], config)
                colliding += bool(collidingSegments(smoother.path, taskExact))
        row[f"smoothed with {checkName}: in collision"] = f"{100 * colliding / (2 * len(tasks)):.0f} %"
    rows.append(row)

pd.DataFrame(rows).set_index("strategy")

Observations:

* With `lineInCollision` about a third of the smoothed paths collide (deterministic strategy: 42 %), although only 2 of the 38 planned paths do. In the example above the path of the extended probabilistic strategy collides at the corner of the "L".
* The deterministic strategy is affected most: it tightens the path most consistently around the obstacles.
* With the exact check no smoothed path collides - here even the two colliding planned paths got collision-free, as their colliding segments were replaced. But this is luck: the smoothing assumes a collision-free path and keeps parts of its segments without checking them again.
* The lecture example above is uncritical: one large obstacle and short shortcuts.

**Remedy**: the shortcuts need a check, which does not miss collisions at the obstacles. `Smoother` takes it from `config`:

* `config["lineCheck"]`: any function `(qa, qb) -> True if in collision`, e.g. `collisionChecker.lineInCollisionExact` - exact (Shapely), but only valid for a **point robot**, where the C-obstacles are the obstacles.
* `config["resolution"]`: the segment is checked at configurations with this maximal distance (`sampledLineCheck` of IPSmoothing.py) - for every collision checker, e.g. for robots. The resolution has to fit the smallest obstacle.

From now on the shortcuts in 2D are checked exactly.

# Comparison of the strategies

As in the lecture, but in 2D: the planned paths from above, every path smoothed 5 times with every strategy, 100 checks each. The quality of a path is its length relative to the shortest path found for this planning task, $l / \min(l)$. (This cell takes some seconds.)

In [ ]:
from IPSmoothing import historySeries

budget, runs = 100, 5

runsPerTask = []
for task in tasks:
    histories = {}
    for strategy in strategies:
        histories[strategy] = []
        for run in range(runs):
            random.seed(run)
            smoother = Smoother(task["benchmark"].collisionChecker)
            smoother.smoothPath(task["path"], {"strategy": strategy, "maxChecks": budget,
                                               "lineCheck": task["benchmark"].collisionChecker.lineInCollisionExact})
            histories[strategy].append((historySeries(smoother.history, budget), smoother.reason))
    task["min"] = min(lengths[-1] for strategy in strategies for (lengths, points), reason in histories[strategy])
    runsPerTask.append(histories)

def quartiles(values):
    q = statistics.quantiles(values, n=4, method="inclusive")
    return q[0], statistics.median(values), q[2]

curves, rows = {}, []
for strategy in strategies:
    relative = [[l / task["min"] for l in lengths] for task, histories in zip(tasks, runsPerTask)
                for (lengths, points), reason in histories[strategy]]
    points = [points for histories in runsPerTask for (lengths, points), reason in histories[strategy]]
    reasons = [reason for histories in runsPerTask for _, reason in histories[strategy]]
    curves[strategy] = {"relative length": [quartiles(c) for c in zip(*relative)],
                        "points": [quartiles(c) for c in zip(*points)]}
    rows.append({"strategy": vis.STRATEGY_NAMES[strategy],
                 **{f"l/min(l) after {c}": round(curves[strategy]["relative length"][c][1], 3) for c in (10, 20, 50, 100)},
                 "points before": statistics.median(p[0] for p in points),
                 "points after 100": statistics.median(p[-1] for p in points),
                 "ended by itself": f"{100 * reasons.count('done') / len(reasons):.0f} %"})

print(f"{len(tasks)} planned paths x {runs} runs per strategy")
vis.showComparison(curves)
pd.DataFrame(rows).set_index("strategy")

Observations:

* After 100 checks all three strategies are within 2 % of the shortest path found: they tend to the **same path quality**.
* The deterministic strategy **converges faster** (after 20 checks: 1.09 against 1.14) and its band is narrower - **smaller variance**. Its only randomness is the planned path.
* The two probabilistic strategies hardly differ here.
* Number of points: the planned paths have many points (K-Closest-PRM up to more than 40), the shortcuts remove most of them. In the long run the probabilistic strategies add points again (every shortcut ends inside a segment). The deterministic one keeps the fewest - in the 6-DOF benchmarks of the lecture it was the other way round, there the bisections added many points.
* The deterministic strategy ended by itself in some runs ($l_A = l_B$): no point could be removed or shortened any more.

# Termination

The loop condition $l_A \neq l_B$ is hardly ever reached by the probabilistic strategies, the deterministic one runs for a long time and adds points. The lecture therefore extends `Smooth` by a criterion `Terminate(p)`: the lengths of the path after the last steps are kept in a queue; as soon as it holds $s_{max}$ values, the oldest one is removed - and the smoothing stops, if the variance of the remaining ones is smaller than $v_{min}$.

```
Terminate(p):            // queue q is kept between the calls
    q.push(length(p))
    if q.size() < s_max: return false
    q.pop()              // remove the oldest element
    return q.variance() < v_min
```

`Smoother` uses it, if `config` contains `sMax` and `vMin` (class `Terminate` in IPSmoothing.py). The same planned paths as above:

In [ ]:
terminateConfig = {"sMax": 10, "vMin": 0.01, "maxChecks": 1000}

rows = []
for strategy in strategies:
    checks, relative, points = [], [], []
    for task in tasks:
        for run in range(runs):
            random.seed(run)
            smoother = Smoother(task["benchmark"].collisionChecker)
            smoother.smoothPath(task["path"], {**terminateConfig, "strategy": strategy,
                                               "lineCheck": task["benchmark"].collisionChecker.lineInCollisionExact})
            checks.append(smoother.checks)
            relative.append(pathLength(smoother.path) / task["min"])
            points.append(len(smoother.path))
    rows.append({"strategy": vis.STRATEGY_NAMES[strategy], "checks (mean)": round(statistics.mean(checks), 1),
                 "l/min(l) (mean)": round(statistics.mean(relative), 3),
                 "l/min(l) (worst)": round(max(relative), 2), "points (median)": statistics.median(points)})

pd.DataFrame(rows).set_index("strategy")

Observations ($s_{max} = 10$, $v_{min} = 0.01$):

* The deterministic strategy stops after about 40 checks, close to the result of 100 checks.
* The probabilistic strategies stop **too early**: after about 26 checks, on average 14 to 17 % longer than the best path, in the worst case twice as long. Some checks without progress do not mean that there is nothing left to gain - successes come by chance.
* So the parameters of `Terminate` have to fit the strategy (and the scale of the C-space, see the robot below).

# What the previous planning tells - and what not

## Effort: planning and smoothing

The smoothing uses the collision checker of the planning. Its effort compared with the planning of the example at the beginning (checked segments counted by `IPPerfMonitor`; the planner uses `lineInCollision`, the smoothing the exact check):

In [ ]:
random.seed(1)
IPPerfMonitor.clearData()
planner = KClosestPRM(benchmark.collisionChecker)
solution = planner.planPath(benchmark.startList, benchmark.goalList, kClosestConfig)
rows = [{"step": "planning (K-Closest-PRM)", "line checks": lineChecks(), "length": round(pathLength(pathFromPlanner(planner, solution)), 1),
         "points": len(solution)}]
for strategy in strategies:
    random.seed(2)
    IPPerfMonitor.clearData()
    smoother = Smoother()
    smoother.smoothSolution(planner, solution, {**terminateConfig, "strategy": strategy, "lineCheck": exact})
    rows.append({"step": f"smoothing ({vis.STRATEGY_NAMES[strategy]}, Terminate)", "line checks": lineChecks(),
                 "length": round(pathLength(smoother.path), 1), "points": len(smoother.path)})

pd.DataFrame(rows).set_index("step")

## The roadmap does not help here

Could the roadmap of the planning give a shorter path without any new check? Its edges are collision-free. But the PRMs of the course only connect nodes of **different** connected components - their roadmap is a forest: between start and goal there is exactly one path. The shortest path by length in the roadmap is the solution itself:

In [ ]:
import networkx as nx

positionsOfNodes = nx.get_node_attributes(planner.graph, "pos")
shortest = nx.shortest_path(planner.graph, "start", "goal",
                            weight=lambda u, v, d: math.dist(positionsOfNodes[u], positionsOfNodes[v]))
print("roadmap is a forest:", nx.is_forest(planner.graph))
print("shortest path by length in the roadmap = solution:", shortest == solution)

So all improvements have to come from new segments - and every new segment has to be checked. This is the reason, why the number of collision checks is the measure for the effort of the smoothing.

## A path from A\*: first remove the collinear points

A\* plans on a grid: its path consists of many short steps, most of the points lie on a straight line. The basic operation **removing collinear points** of the slides needs no collision check - the path does not change, only the points on straight lines between their neighbours are dropped ($Q = 1$). A\* has only checked the grid points with `pointInCollision`, not the segments between them; the smoothing checks every new segment.

In [ ]:
from IPAStar import AStar
from IPSmoothing import removeCollinear

astar = AStar(benchmark.collisionChecker)
astarSolution = astar.planPath(benchmark.startList, benchmark.goalList, {"w": 0.5, "heuristic": "euclidean"})
astarPath = pathFromPlanner(astar, astarSolution)
reduced = removeCollinear(astarPath)
print(f"A*: {len(astarPath)} points, after removing the collinear ones {len(reduced)} points, "
      f"length {pathLength(astarPath):.1f} (unchanged: {pathLength(reduced):.1f})")

results = []
for title, start in (("A*, smoothed", astarPath), ("A*, collinear points removed, smoothed", reduced)):
    smoother = Smoother(benchmark.collisionChecker)
    smoothed = smoother.smoothPath(start, {**terminateConfig, "strategy": "deterministic", "lineCheck": exact})
    results.append((f"{title} ({smoother.checks} checks)", smoother, smoothed))
vis.showResults(results, panelWidth=5.0)

Observations: removing the collinear points costs nothing and the smoothing then needs fewer checks for the same result - every collinear point would have been a candidate for a shortcut, which brings nothing.

# Robots: smoothing in the joint space

The smoothing only needs a check of segments - it works in any C-space. IP-10-0 ended with the observation, that the planned motions of the planar robot are valid, but not nice: the robot swings around, because the path follows random nodes. This chapter continues there, with the same robot, scene, start and goal.

* The path is smoothed in the **joint space**; a "segment" is a synchronous motion of all joints.
* For a robot there is no exact check of a segment: `KinChainCollisionChecker.lineInCollision` checks configurations with the distance `fk_resolution` in the joint space (IP-10-0, "Lines in the joint space").
* The figures need the C-obstacles: `computeCSpace()` samples them (only for the figures, 300 x 300 collision checks - a coarser grid does not show their thin tips). Afterwards `smoothingVisualize` draws them like the visualizations of the planners.

`robotProblem` builds the robot and its collision checker as in IP-10-0, `planAndSmooth` runs the Lazy-PRM and the smoothing with the **same** collision checker and counts the collision checks of both.

In [ ]:
import numpy as np
from shapely.geometry import LineString

import IPVISPlanarManipulator as robotVis
from IPEnvironmentKin import KinChainCollisionChecker, animateSolution
from IPLazyPRM import LazyPRM
from IPPlanarManipulator import PlanarRobot
from IPSmoothing import sampledLineCheck

obstacles = dict()
obstacles["obs1"] = LineString([(-2, 0), (-0.8, 0)]).buffer(0.5)
obstacles["obs2"] = LineString([(2, 0), (2, 1)]).buffer(0.2)
obstacles["obs3"] = LineString([(-1, 2), (1, 2)]).buffer(0.1)

lazyConfig = {"initialRoadmapSize": 40, "updateRoadmapSize": 20, "kNearest": 8, "maxIterations": 40}   # as in IP-10-0
robotConfig = {**terminateConfig, "vMin": 0.0001}


def robotProblem(n=2, fkResolution=0.2, linkRadius=0.0):
    "Planar robot with n joints and total length 3 (IP-10-0): returns (collision checker, start, goal)"
    robot = PlanarRobot(n_joints=n)
    for joint in robot.joints:
        joint.a = 3.0 / n
    checker = KinChainCollisionChecker(robot, obstacles, limits=[[-math.pi, math.pi]] * n,
                                       fk_resolution=fkResolution, link_radius=linkRadius)
    return checker, [2.0] + [0.5 / (n - 1)] * (n - 1), [-2.0] + [-0.5 / (n - 1)] * (n - 1)


def pointChecks():
    # calls of pointInCollision since the last IPPerfMonitor.clearData()
    calls = IPPerfMonitor.dataFrame()
    return int((calls["name"] == "pointInCollision").sum()) if len(calls) else 0


def planAndSmooth(checker, start, goal, seed=0, strategy="deterministic"):
    # Lazy-PRM and smoothing with the same collision checker
    # returns (planned path, smoother, solution of the smoother, checks of planning, checks of smoothing)
    random.seed(seed)
    IPPerfMonitor.clearData()
    lazy = LazyPRM(checker)
    lazySolution = lazy.planPath([start], [goal], lazyConfig)
    if not lazySolution:
        return None
    planningChecks = pointChecks()
    random.seed(seed)
    IPPerfMonitor.clearData()
    smoother = Smoother()
    smoothedSolution = smoother.smoothSolution(lazy, lazySolution, {**robotConfig, "strategy": strategy})
    return pathFromPlanner(lazy, lazySolution), smoother, smoothedSolution, planningChecks, pointChecks()

## The planned and the smoothed motion

Two joints, links as lines, `fk_resolution = 0.02`. Left the joint space with the C-obstacles: the path of the Lazy-PRM (dashed) and the smoothed path (blue). In the joint space the lengths are much smaller than in the benchmarks (radian instead of the units of the benchmarks), therefore $v_{min}$ of `Terminate` is smaller here (`robotConfig`): $v_{min}$ depends on the scale of the C-space.

In [ ]:
fineChecker, robotStart, robotGoal = robotProblem(2, fkResolution=0.02)
fineChecker.computeCSpace(300)      # only for the figures

results, plannedPath = [], None
for strategy in ("latombe", "deterministic"):
    plannedPath, robotSmoother, robotSolution, planningChecks, smoothingChecks = planAndSmooth(
        fineChecker, robotStart, robotGoal, seed=0, strategy=strategy)
    results.append((vis.STRATEGY_NAMES[strategy], robotSmoother, robotSolution))
    print(f"{vis.STRATEGY_NAMES[strategy]:36s} point checks: planning {planningChecks}, smoothing {smoothingChecks}")

vis.showResults(results, panelWidth=5.0)

The same in the **work space** (start green, goal red, blue: the path of the tip) and as joint angles over the path:

In [ ]:
smoothedPath = robotSmoother.path     # deterministic strategy

robotVis.showMotions(fineChecker, [("planned by the Lazy-PRM", plannedPath), ("smoothed", smoothedPath)])
robotVis.showJointAngles([plannedPath, smoothedPath], titles=["planned", "smoothed"])

pd.DataFrame([{"path": name, "points": len(p), "length in the joint space [rad]": round(pathLength(p), 2),
               "path of the tip in the work space": round(robotVis.tcpPathLength(fineChecker, p), 2)}
              for name, p in (("planned", plannedPath), ("smoothed", smoothedPath))]).set_index("path")

Observations:

* In the joint space the path gets clearly shorter: the joints do not move back and forth any more (see the joint angles).
* The path of the **tip** in the work space gets only a little shorter. The smoothing minimizes the length in the C-space, i.e. the motion of the joints - not the way of the tool. What "short" means, is defined by the metric of the C-space.
* The smoothed path runs along the C-obstacles: in the work space the robot passes the obstacles as closely as possible.

## Smoothing needs a reliable check - even more than planning

IP-10-0 showed, that with `fk_resolution = 0.2` one of ten paths of the Lazy-PRM leads through an obstacle. The smoothing makes this worse:

* it **keeps parts of the planned segments** without a new check - it assumes a collision-free path,
* it **tightens** the path around the C-obstacles: the new segments run exactly, where a sampled check misses collisions.

Ten runs per setting. Every path is checked independently with the distance 0.005 against the real robot (links as lines). The third setting uses a **safety margin**: planning and smoothing use a robot with capsule links of radius 0.05 (`link_radius`, see "Links with a thickness" in IP-10-0), the result is checked against the real robot.

In [ ]:
realChecker, _, _ = robotProblem(2)
fine = sampledLineCheck(realChecker, 0.005)      # independent check against the real robot


def valid(path):
    return len(collidingSegments(path, fine)) == 0


settings = [("fk_resolution 0.2", 0.2, 0.0), ("fk_resolution 0.02", 0.02, 0.0),
            ("fk_resolution 0.05, margin 0.05", 0.05, 0.05)]
rows, examples = [], dict()
for name, fkResolution, margin in settings:
    checker, _, _ = robotProblem(2, fkResolution, linkRadius=margin)
    row = {"setting": name, "planned valid": 0}
    for strategy in ("latombe", "deterministic"):
        validPaths, checks, lengths = 0, [], []
        for seed in range(10):
            planned, smoother, solution, planningChecks, smoothingChecks = planAndSmooth(
                checker, robotStart, robotGoal, seed, strategy)
            validPaths += valid(smoother.path)
            checks.append(smoothingChecks)
            lengths.append(pathLength(smoother.path))
            if strategy == "deterministic":
                row["planned valid"] += valid(planned)
                if seed == 1:
                    examples[name] = (checker, smoother, solution)
        row[f"{strategy}: smoothed valid"] = validPaths
        row[f"{strategy}: point checks"] = int(statistics.mean(checks))
        row[f"{strategy}: length"] = round(statistics.mean(lengths), 2)
    rows.append(row)

pd.DataFrame(rows).set_index("setting")

Observations:

* `fk_resolution = 0.2`: nine of the ten planned paths are valid - but only five of the paths smoothed with the extended probabilistic strategy and **none** of the paths smoothed with the deterministic strategy. The deterministic strategy tightens the path most, so it is affected most.
* `fk_resolution = 0.02`: about eight times as many collision checks, and still two of the ten deterministically smoothed paths graze an obstacle. A path, which touches the C-obstacles, cannot be made safe by a finer sampling alone.
* **Safety margin**: all paths are valid - with less than half of the collision checks of the fine resolution. The price is a slightly longer path (about 1 %).
* As in IP-10-0: the result has to be verified independently of the planner and the smoother.

Two examples of the deterministic strategy (seed 1):

In [ ]:
for name in ("fk_resolution 0.2", "fk_resolution 0.05, margin 0.05"):
    examples[name][0].computeCSpace(300)      # only for the figures

vis.showResults([(name, smoother, solution) for name, (checker, smoother, solution) in examples.items()
                 if name != "fk_resolution 0.02"], panelWidth=5.0, check=fine)

Left (`fk_resolution = 0.2`, seed 1): the planned path is valid, the smoothed one cuts the tip of a C-obstacle (orange, dashed) - in an animation the robot would move through a thin obstacle. Right: with the safety margin the C-obstacles are a bit larger (they belong to the thicker robot), and the smoothed path keeps a distance from the real obstacles.

## More joints

The smoothing works in the same way for more joints - only the figures of the joint space are not possible any more. Robots with 2 to 8 joints as in IP-10-0 (links with radius 0.05), planned and smoothed with a safety margin of 0.02 and `fk_resolution = 0.05`; five runs each.

In [ ]:
MARGIN = 0.02

rows, motions = [], dict()
for n in [2, 3, 4, 6, 8]:
    realCheckerN, startN, goalN = robotProblem(n, 0.05, linkRadius=0.05)
    safeChecker, _, _ = robotProblem(n, 0.05, linkRadius=0.05 + MARGIN)
    fineN = sampledLineCheck(realCheckerN, 0.005)
    runs = []
    for seed in range(5):
        result = planAndSmooth(safeChecker, startN, goalN, seed)
        if result is None:
            continue
        planned, smoother, solution, planningChecks, smoothingChecks = result
        runs.append({"planned": pathLength(planned), "smoothed": pathLength(smoother.path),
                     "tip planned": robotVis.tcpPathLength(realCheckerN, planned),
                     "tip smoothed": robotVis.tcpPathLength(realCheckerN, smoother.path),
                     "planning": planningChecks, "smoothing": smoothingChecks,
                     "valid": len(collidingSegments(smoother.path, fineN)) == 0})
        if seed == 1:
            motions[n] = (realCheckerN, planned, smoother.path)
    runs = pd.DataFrame(runs)
    rows.append({"joints": n, "paths": len(runs), "smoothed valid": int(runs["valid"].sum()),
                 "length planned [rad]": round(runs["planned"].mean(), 1),
                 "length smoothed [rad]": round(runs["smoothed"].mean(), 1),
                 "straight line [rad]": round(math.dist(startN, goalN), 1),
                 "tip planned": round(runs["tip planned"].mean(), 1),
                 "tip smoothed": round(runs["tip smoothed"].mean(), 1),
                 "point checks planning": int(runs["planning"].mean()),
                 "point checks smoothing": int(runs["smoothing"].mean())})

pd.DataFrame(rows).set_index("joints")

Observations:

* All smoothed paths are valid (checked against the robot without margin).
* The paths of the Lazy-PRM get longer with every joint (7.9 rad with two, 15.3 rad with eight joints), because every random node moves all joints. The smoothing removes a fifth (two joints) to a third (eight joints) of the length; the rest is needed to get around the C-obstacles (straight line: 4.0 rad).
* The path of the tip in the work space hardly changes - the smoothing calms the **joints**, see the joint angles below.
* Here the smoothing needs about as many collision checks as the planning: the Lazy-PRM is a very cheap planner in these mostly free C-spaces, and every check of a shortcut samples a long segment with a fine resolution.

The robot with eight joints (seed 1):

In [ ]:
checker8, planned8, smoothed8 = motions[8]
robotVis.showMotions(checker8, [("eight joints: planned by the Lazy-PRM", planned8), ("smoothed", smoothed8)])
robotVis.showJointAngles([planned8, smoothed8], titles=["planned", "smoothed"])

## Animation

(Optional, takes some seconds.) `animateSolution` of IPEnvironmentKin.py works with the smoother like with a planner: left the robot in the work space, right the smoothed path in the joint space (two joints, the example with the safety margin from above).

In [ ]:
marginChecker, marginSmoother, marginSolution = examples["fk_resolution 0.05, margin 0.05"]
animateSolution(marginSmoother, marginChecker, marginSolution, smoothingVisualize, workSpaceLimits=[[-3.3, 3.3], [-3.3, 3.3]])

# Properties of the smoothing strategies

* **Only path and collision checker needed**: the smoothing works with the result of every planner and in every C-space (benchmarks in 2D, joint space of a robot with any number of joints).
* **Cheap compared with the planning**: one line check per step - in the example some 20 to 50 checks against more than 300 for the planning, and the path gets a third to 40 % shorter.
* **Pure probabilistic**: simple, without memory; many shortcuts collide or bring little.
* **Extended probabilistic (Latombe)**: a colliding shortcut is not discarded but shortened towards $l_{rand}$.
* **Deterministic (Berchtold & Glavina)**: treats the worst point first; converges fastest, with the smallest variance; may end by itself.
* **Termination**: $l_A = l_B$ is hardly ever reached; `Terminate` has to fit the strategy and the scale of the C-space.
* **Reliable check of the shortcuts**: the smoothing moves the path to the obstacles - exactly where sampled checks miss collisions. A point robot is checked exactly. For a robot a fine resolution alone is expensive and still not safe; a **safety margin** (a slightly thicker robot for planning and smoothing) is cheaper and gives valid paths. And the path given to the smoothing has to be collision-free: parts of its segments are kept without a new check.
* **Short in the C-space**: the smoothing shortens the path in the metric of the C-space (for a robot: the motion of the joints), not the way of the tool in the work space.
* The smoothed path is **tightened like a rubber band**: it touches the obstacles and a robot has to stop at every path point. Rounding of the movements (next topic of the lecture) needs a modified path.

# Experiments and Exercises

1. Implement the operation **DelEqual** of the slides as an alternative bisection: the new point on the longer segment gets the distance of the shorter segment from the worst point. Compare it with DelTree on the benchmarks.
2. Implement the **Mid-Point** approach (replace $p_{i+1}$ by the midpoints $z_i$, $z_{i+1}$, if the segment between them is collision-free) and compare the number of path points with the generic approach.
3. Choose $s_{max}$ and $v_{min}$ of `Terminate` such that the probabilistic strategies reach the same path quality as with 100 checks. How many checks do they need? Is a relative criterion (variance divided by the squared length) better?
4. Robot: find combinations of `fk_resolution` and safety margin, for which all smoothed paths are valid when checked with the distance 0.005. Which combination needs the fewest calls of `pointInCollision`? How does the necessary margin depend on the resolution and on the length of the links?
5. The deterministic strategy finds the worst point by a search over all points. Use a priority list (`heapq`) instead. Which ratings have to be updated after a successful shortcut?
6. Smooth the paths of the Visibility-PRM (IP-6-0) and of the BB-Method (IP-11-0). Which planner profits most?
7. (Advanced) A smoothed path touches the obstacles. Smooth the benchmarks with a **safety distance**: a collision checker, whose obstacles are grown by a distance $s$ (`buffer`). What happens to the path length?
8. (Advanced) The smoothing shortens the path in the joint space. Weight the joints in the path length (e.g. the first joint more than the last one) or rate a path by the way of the tip (`robotVis.tcpPathLength`). How does the smoothed motion change?